# Oil Prices in 2026 — Act 5: What If the Model Could Read the News?

The [companion notebook](energy_oil_case_study.ipynb) showed that Prophet's rolling 30-day
forecast catastrophically missed the 2026 oil price surge — forecasting ~$61/bbl
while WTI hit $100. The model wasn't wrong in principle; it simply had no mechanism
for incorporating the geopolitical context that was publicly available at the time.

This notebook asks: **could a context-aware LLM forecaster have done better?**

We evaluate three key forecast origins in early 2026 using three methods side by side:
- **Prophet** (baseline — already computed, loaded from cache)
- **LLMP — no context** (Gemini 3 Flash, history only)
- **LLMP — with context** (same model + plausibly-knowable geopolitical context at each origin)

And we frame the comparison three ways:

| | Question type | Evaluation |
|---|---|---|
| **Act 5** | *Trajectory* — what will the 30-day price path look like? | MAE vs. actuals |
| **Act 6** | *Binary* — will price exceed a meaningful threshold in 30 days? | Calibration of P(exceed) |
| **Act 7** | *Causal* — what forces are the model anchoring on? | Qualitative reasoning audit |

In [ ]:
from __future__ import annotations

import json
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as psp
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)

# ── Repo root: walk up from CWD until pyproject.toml is found ─────────────────
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = _cwd
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"

for p in [str(REPO_ROOT / "implementations"), str(REPO_ROOT / "aieng-forecasting")]:
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(REPO_ROOT / ".env")

# ── Colour palette (matches companion notebook) ────────────────────────────────
CLR_HISTORY   = "#bdd7e7"
CLR_ACTUAL    = "#2171b5"   # solid blue — the truth
CLR_PROPHET   = "#636363"   # grey — the blind baseline
CLR_LLMP_BARE = "#fd8d3c"   # orange — LLM, history only
CLR_LLMP_CTX  = "#2ca02c"   # green — LLM, with context
CLR_CONFLICT  = "#d62728"   # red — conflict annotation
CONFLICT_DATE = pd.Timestamp("2026-03-01")

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print("Setup complete.")

In [ ]:
# ── WTI price history (same cache as companion notebook) ──────────────────────
PRICE_CACHE    = DATA_DIR / "wti_price_history.parquet"
PROPHET_CACHE  = DATA_DIR / "energy_case_study_forecasts_30d_daily_v3.parquet"
LLMP_CACHE     = DATA_DIR / "energy_llmp_context_forecasts.parquet"

price_df = pd.read_parquet(PRICE_CACHE)
price_df.index = pd.DatetimeIndex(
    [pd.Timestamp(str(d)[:10]) for d in price_df.index]
)
price_df.index.name = "date"
price_df = price_df.sort_index()

prophet_df = pd.read_parquet(PROPHET_CACHE)
prophet_df["sim_day"]        = pd.to_datetime(prophet_df["sim_day"])
prophet_df["resolution_date"] = pd.to_datetime(prophet_df["resolution_date"])

print(f"WTI price history : {price_df.index[0].date()} → {price_df.index[-1].date()} ({len(price_df):,} days)")
print(f"Prophet forecasts : {prophet_df['sim_day'].min().date()} → {prophet_df['sim_day'].max().date()} ({len(prophet_df):,} rows)")

---

## The Setup

We pick **three forecast origins** in early 2026 — each representing a different
stage of the geopolitical escalation that drove WTI from ~$58 to $100+ between
January and April 2026.

| Origin | WTI at origin | Resolution date | Actual WTI at resolution | Prophet forecast | Context available |
|---|---|---|---|---|---|
| Jan 5, 2026 | $58 | Feb 4, 2026 | **$65** | $58 (inside CI) | Tensions building; OPEC+ cuts; insurance premiums rising |
| Feb 2, 2026 | $62 | Mar 4, 2026 | **$75** | $61 (miss — above CI) | Gulf of Oman incident; escalation fears; analyst upgrades |
| Mar 2, 2026 | $71 | Apr 1, 2026 | **$100** | $61 (catastrophic miss) | Conflict active; Strait of Hormuz blockade; IEA emergency session |

For each origin we ask: does an LLMP with access to publicly-available context
shift its forecast in the right direction — toward the actual outcome?

In [ ]:
def compress_history(
    price_df: pd.DataFrame,
    as_of: pd.Timestamp,
    recent_window_months: int = 6,
) -> pd.DataFrame:
    """Return a token-efficient history: weekly averages for older data, daily for recent.

    Compresses ~1300 daily rows to ~200 rows while preserving the recent
    daily granularity that matters most for the LLM's short-horizon forecast.
    """
    hist = price_df[price_df.index <= as_of].copy()
    cutoff_daily = as_of - pd.DateOffset(months=recent_window_months)

    older = (
        hist[hist.index < cutoff_daily]
        .resample("W")
        .mean()
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )
    recent = (
        hist[hist.index >= cutoff_daily]
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )

    result = pd.concat([older, recent], ignore_index=True)
    result["timestamp"] = pd.to_datetime(result["timestamp"])
    return result[["timestamp", "value"]].sort_values("timestamp").reset_index(drop=True)


def prophet_row_at_origin(prophet_df: pd.DataFrame, origin: pd.Timestamp) -> pd.Series:
    """Return the Prophet forecast row whose sim_day is nearest to (on or after) origin."""
    candidates = prophet_df[prophet_df["sim_day"] >= origin]
    return candidates.iloc[0]


def resolution_price(price_df: pd.DataFrame, origin: pd.Timestamp, horizon_calendar_days: int = 30) -> tuple[pd.Timestamp, float]:
    """Return (resolution_date, actual_price) for a calendar-day horizon from origin."""
    target = origin + pd.Timedelta(days=horizon_calendar_days)
    row = price_df[price_df.index >= target].iloc[0]
    return row.name, float(row["price"])


# Sanity-check the three origins
ORIGINS = [
    pd.Timestamp("2026-01-05"),
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-03-02"),
]

print("Origin summary:")
for o in ORIGINS:
    price_at_origin = float(price_df[price_df.index >= o].iloc[0]["price"])
    res_date, res_price = resolution_price(price_df, o)
    p_row = prophet_row_at_origin(prophet_df, o)
    print(
        f"  {o.date()}  WTI=${price_at_origin:.2f}  "
        f"→ resolution {res_date.date()} actual=${res_price:.2f}  "
        f"prophet=${p_row['yhat']:.2f} [{p_row['yhat_lower']:.1f},{p_row['yhat_upper']:.1f}]  "
        f"inside_ci={p_row['inside_ci']}"
    )

In [ ]:
# ── Context snippets — plausibly knowable on each origin date ─────────────────
# These represent the kind of information a professional energy analyst
# would have had access to from public sources: news, vessel-tracking
# services, futures data, and analyst reports published before the origin date.

ORIGIN_CONTEXTS: dict[str, dict] = {
    "2026-01-05": {
        "label": "Jan 5, 2026",
        "threshold_usd": 65.0,
        "context_text": (
            "As of January 5 2026:\n"
            "- WTI crude has been range-bound in the $56–62 band since October 2025 on soft "
            "demand signals and elevated US inventory builds.\n"
            "- OPEC+ is maintaining its current production-cut agreement through Q1 2026; "
            "no rollback has been signalled.\n"
            "- Iranian proxy forces conducted three separate attacks on US logistics assets "
            "in Iraq and Syria in Q4 2025. US-Iran tensions are elevated but have not "
            "escalated to direct military exchange.\n"
            "- Lloyd's of London hull-war insurance premiums for tankers transiting the "
            "Gulf of Oman have risen approximately 15% since September 2025.\n"
            "- The WTI NYMEX forward curve is in mild backwardation: front month $58, "
            "6-month forward approximately $56.\n"
            "- EIA weekly report (Dec 31 2025): US crude inventories 8% below the 5-year "
            "seasonal average."
        ),
    },
    "2026-02-02": {
        "label": "Feb 2, 2026",
        "threshold_usd": 72.0,
        "context_text": (
            "As of February 2 2026:\n"
            "- WTI gained approximately 7% in January, closing near $62, driven by "
            "escalating Persian Gulf tensions.\n"
            "- A US Navy escort mission in the Gulf of Oman was intercepted by Iranian "
            "fast-attack boats on January 28. No shots fired, but the incident was "
            "widely reported and prompted a diplomatic protest from Washington.\n"
            "- OPEC+ called an emergency ministerial consultation for February 10 amid "
            "concerns about supply-chain disruption risk; no production change announced yet.\n"
            "- Goldman Sachs revised its 2026 WTI price target upward to $70–85 in a "
            "February 1 research note, citing a 'geopolitical risk premium re-rating'.\n"
            "- Vessel-tracking data shows tanker transits through the Strait of Hormuz "
            "down approximately 15% week-over-week, as operators seek alternative routings.\n"
            "- Brent/WTI spread widened to $4.50, the largest since early 2024, as "
            "European buyers began bidding up non-Gulf grades.\n"
            "- US intelligence officials stated publicly that Iranian military assets "
            "have been repositioned closer to the Strait of Hormuz."
        ),
    },
    "2026-03-02": {
        "label": "Mar 2, 2026",
        "threshold_usd": 85.0,
        "context_text": (
            "As of March 2 2026:\n"
            "- The US conducted direct airstrikes on Iranian oil-infrastructure targets "
            "on March 1 2026 in response to an Iranian attack on a US carrier group "
            "in the Gulf of Oman on February 26.\n"
            "- Iran declared a partial blockade of the Strait of Hormuz effective "
            "March 1; approximately 20% of global seaborne oil supply transits the Strait.\n"
            "- WTI surged from $62 on February 2 to $71 by March 2 — a 14% move in "
            "one month — and front-month futures gapped up a further $4 at Monday open.\n"
            "- The IEA called an emergency ministerial meeting for March 5 to consider "
            "releasing strategic petroleum reserves.\n"
            "- Saudi Aramco issued force majeure declarations on several customer contracts; "
            "Saudi Arabia activated its emergency supply protocols.\n"
            "- Goldman Sachs issued an updated note on March 1 with a new 2026 WTI target "
            "of $95–115 and flagging a tail-risk scenario of $130 if the blockade persists "
            "beyond 60 days.\n"
            "- WTI NYMEX forward curve has swung into sharp backwardation: front month "
            "$71, 6-month forward $62, signalling market expectation of eventual resolution."
        ),
    },
}

print("Context snippets defined for:", list(ORIGIN_CONTEXTS.keys()))

In [ ]:
# ── Run LLMP forecasts (or load from cache) ───────────────────────────────────
#
# We use the LLMP module's internals directly so we can supply a pre-compressed
# history DataFrame rather than going through a DataService.  This is appropriate
# for a playground notebook; a production version would use a registered adapter.
#
# Each origin × 2 variants (bare / with-context) = 6 LLM calls.
# Results are cached to data/energy_llmp_context_forecasts.parquet.

from aieng.forecasting.methods.llm_processes.continuous import (
    ContinuousLLMPredictorConfig,
    _build_system_prompt,
    _build_user_prompt,
    _quantiles_per_step,
    _sample_trajectories,
    _stack_trajectories,
)
from aieng.forecasting.methods.llm_processes.base import serialize_history
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.data.models import SeriesMetadata


MODEL       = "gemini/gemini-3-flash-preview"
N_SAMPLES   = 20
HORIZON_B   = 21   # ~21 business days ≈ 30 calendar days
PRECISION   = 2

_WTI_TASK = ForecastingTask(
    task_id="wti_crude_30d",
    target_series_id="wti_crude",
    horizons=list(range(1, HORIZON_B + 1)),
    frequency="B",
    description=(
        "WTI crude oil front-month futures price (USD/bbl), "
        "30 trading-day ahead probabilistic forecast. "
        "Forecast the daily closing price for each of the next "
        f"{HORIZON_B} business days."
    ),
)

_WTI_META = SeriesMetadata(
    series_id="wti_crude",
    description="WTI crude oil front-month futures (CL=F, Adj Close)",
    source="Yahoo Finance",
    units="USD/bbl",
    frequency="B",
)


def _run_one_forecast(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    context_text: str | None,
    context_tag: str,
) -> dict:
    """Run LLMP at a single origin and return a dict of arrays."""
    history_df = compress_history(price_df, origin)
    history_str = serialize_history(history_df, precision=PRECISION)

    forecast_start = origin + pd.offsets.BDay(1)
    forecast_end   = origin + pd.offsets.BDay(HORIZON_B)

    system_prompt = _build_system_prompt()
    user_prompt   = _build_user_prompt(
        _WTI_TASK, history_str, _WTI_META,
        forecast_start, forecast_end, HORIZON_B,
        context_text=context_text,
    )

    cfg = ContinuousLLMPredictorConfig(
        model=MODEL,
        n_samples=N_SAMPLES,
        temperature=1.0,
        reasoning_effort="disable",
        context_text=context_text,
        context_tag=context_tag,
    )

    parsed, cost_usd, in_tok, out_tok, failures = _sample_trajectories(
        cfg=cfg, system_prompt=system_prompt, user_prompt=user_prompt
    )
    samples = _stack_trajectories([t.values for t in parsed], n_steps=HORIZON_B)
    q_grid  = _quantiles_per_step(samples)   # (HORIZON_B, len(STANDARD_QUANTILES))

    dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=HORIZON_B)

    rows = []
    for h_idx in range(HORIZON_B):
        row: dict = {
            "origin":      origin,
            "context_tag": context_tag,
            "forecast_date": dates[h_idx],
            "horizon":     h_idx + 1,
            "median":      float(q_grid[h_idx, STANDARD_QUANTILES.index(0.50)]),
            "cost_usd":    cost_usd,
        }
        for qi, q in enumerate(STANDARD_QUANTILES):
            row[f"q{int(q * 100):02d}"] = float(q_grid[h_idx, qi])
        rows.append(row)

    print(
        f"    origin={origin.date()} tag={context_tag:12s} "
        f"cost=${cost_usd:.4f}  failures={failures}/{N_SAMPLES}"
    )
    return {"rows": rows, "samples": samples.tolist()}


def run_all_forecasts(price_df: pd.DataFrame, cache_path: Path) -> pd.DataFrame:
    """Run (or load) all 6 LLMP forecasts; return a single flat DataFrame."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} LLMP forecast rows from cache.")
        return df

    all_rows: list[dict] = []
    print("Running LLMP forecasts (6 API calls)...")
    for origin in ORIGINS:
        key = origin.strftime("%Y-%m-%d")
        ctx = ORIGIN_CONTEXTS[key]
        for tag, text in [("bare", None), ("context", ctx["context_text"])]:
            result = _run_one_forecast(price_df, origin, text, tag)
            all_rows.extend(result["rows"])

    df = pd.DataFrame(all_rows)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


llmp_df = run_all_forecasts(price_df, LLMP_CACHE)
print(f"\nOrigins: {sorted(llmp_df['origin'].dt.date.unique())}")
print(f"Tags:    {sorted(llmp_df['context_tag'].unique())}")
llmp_df.head(6)

---

## Act 5 — Trajectory: Can the Model See the Move Coming?

Each panel below shows the 30-day forecast fan from one origin date.
Three forecast traces:
- **Grey** — Prophet (statistical baseline; history only)
- **Orange** — LLMP, history only (same information as Prophet, different model family)
- **Green** — LLMP with context (public geopolitical information available on that date)

The **solid blue line** is the realized WTI price (actual outcome).
Shading shows 50% and 90% credible intervals for the LLMP forecasts.

In [ ]:
def make_trajectory_figure(
    price_df: pd.DataFrame,
    prophet_df: pd.DataFrame,
    llmp_df: pd.DataFrame,
    origins: list[pd.Timestamp],
) -> go.Figure:
    """Three-column subplot: one panel per origin, trajectory fan comparison."""
    labels = [ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in origins]

    fig = psp.make_subplots(
        rows=1, cols=3,
        subplot_titles=labels,
        shared_yaxes=False,
        horizontal_spacing=0.06,
    )

    for col, origin in enumerate(origins, start=1):
        key = origin.strftime("%Y-%m-%d")

        # History (60 days pre-origin)
        hist_start = origin - pd.Timedelta(days=60)
        hist = price_df[price_df.index >= hist_start].loc[:origin]
        fig.add_trace(
            go.Scatter(
                x=hist.index, y=hist["price"],
                line=dict(color=CLR_HISTORY, width=2),
                name="History" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="history",
            ),
            row=1, col=col,
        )

        # Actuals post-origin
        res_date, _ = resolution_price(price_df, origin)
        actuals = price_df[(price_df.index > origin) & (price_df.index <= res_date)]
        fig.add_trace(
            go.Scatter(
                x=actuals.index, y=actuals["price"],
                line=dict(color=CLR_ACTUAL, width=2.5),
                name="Actual" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="actual",
            ),
            row=1, col=col,
        )

        # Prophet CI band + median
        p_row = prophet_row_at_origin(prophet_df, origin)
        fig.add_trace(
            go.Scatter(
                x=[p_row["resolution_date"]] * 2,
                y=[p_row["yhat_lower"], p_row["yhat_upper"]],
                mode="markers",
                marker=dict(symbol="line-ew", size=14, color=CLR_PROPHET, line=dict(width=2, color=CLR_PROPHET)),
                name="Prophet 95% CI" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="prophet_ci",
            ),
            row=1, col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=[origin, p_row["resolution_date"]],
                y=[p_row["yhat"], p_row["yhat"]],
                line=dict(color=CLR_PROPHET, width=1.5, dash="dot"),
                name="Prophet median" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="prophet_med",
            ),
            row=1, col=col,
        )

        # LLMP fans (bare + context)
        for tag, clr, name in [
            ("bare",    CLR_LLMP_BARE, "LLMP — history only"),
            ("context", CLR_LLMP_CTX,  "LLMP — with context"),
        ]:
            sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag)].sort_values("forecast_date")
            if sub.empty:
                continue

            # 90% CI shading
            fig.add_trace(
                go.Scatter(
                    x=pd.concat([sub["forecast_date"], sub["forecast_date"][::-1]]),
                    y=pd.concat([sub["q05"], sub["q95"][::-1]]),
                    fill="toself",
                    fillcolor=clr,
                    opacity=0.10,
                    line=dict(width=0),
                    showlegend=False,
                ),
                row=1, col=col,
            )
            # 50% CI shading
            fig.add_trace(
                go.Scatter(
                    x=pd.concat([sub["forecast_date"], sub["forecast_date"][::-1]]),
                    y=pd.concat([sub["q25"], sub["q75"][::-1]]),
                    fill="toself",
                    fillcolor=clr,
                    opacity=0.20,
                    line=dict(width=0),
                    showlegend=False,
                ),
                row=1, col=col,
            )
            # Median line
            fig.add_trace(
                go.Scatter(
                    x=sub["forecast_date"], y=sub["median"],
                    line=dict(color=clr, width=2),
                    name=name if col == 1 else None,
                    showlegend=(col == 1),
                    legendgroup=f"llmp_{tag}",
                ),
                row=1, col=col,
            )

        # Origin marker
        fig.add_vline(
            x=origin.timestamp() * 1000,
            line=dict(color="#888", dash="dash", width=1),
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text="30-Day WTI Forecast Trajectories — Prophet vs. LLMP vs. LLMP + Context",
            font=dict(size=15),
        ),
        height=420,
        width=1200,
        legend=dict(orientation="h", y=-0.18),
        template="plotly_white",
        margin=dict(t=60, b=90),
    )
    fig.update_yaxes(title_text="WTI (USD/bbl)", col=1)
    return fig


make_trajectory_figure(price_df, prophet_df, llmp_df, ORIGINS).show()

### What to look for

- **January origin**: Prophet barely covers the outcome (within CI). Does LLMP-with-context
  shift the median upward given the rising-tension signals? A modest upward shift is the
  "right" answer — the context signals elevated risk without a clear catalyst yet.
- **February origin**: Prophet misses (actual $75, CI upper ~$69). Does LLMP-with-context
  produce a median meaningfully higher than LLMP-bare? The context describes a naval incident
  and analyst upgrades — strong reasons to expect further upside.
- **March origin**: The most dramatic case. Conflict is active. Does LLMP-with-context
  shift its distribution toward $100? Anything in the $80–95 range would represent a
  major improvement over Prophet's $61.

---

## Act 6 — Binary: P(Price > Threshold in 30 Days)

The same forecasts, reframed as a probability-of-exceeding question.

This is what most business decisions actually care about: not *exactly* what price will be,
but *whether it will cross a decision-relevant threshold*.

| Origin | Threshold | Why this threshold |
|---|---|---|
| Jan 5 | **\$65** | Just above Prophet's CI upper; tests whether context raises the right tail |
| Feb 2 | **\$72** | Above Prophet's CI upper ($69); actual outcome was $75 |
| Mar 2 | **\$85** | Well above Prophet's range; meaningful supply-disruption price level |

Evaluated with **Brier score**: lower is better, 0 = perfect, 0.25 = climatology baseline.

In [ ]:
import scipy.interpolate


def prob_above_threshold(
    llmp_sub: pd.DataFrame,
    threshold: float,
) -> float:
    """Estimate P(30d-ahead price > threshold) from the LLMP quantile grid at horizon=21."""
    row = llmp_sub[llmp_sub["horizon"] == HORIZON_B]
    if row.empty:
        row = llmp_sub.iloc[-1:]
    row = row.iloc[0]

    q_levels = STANDARD_QUANTILES
    q_vals   = [float(row[f"q{int(q * 100):02d}"]) for q in q_levels]

    # Sort by value (defensive — should already be monotone)
    pairs = sorted(zip(q_vals, q_levels))
    vals, probs = zip(*pairs)

    cdf_at_thr = float(
        scipy.interpolate.interp1d(
            vals, probs, kind="linear", bounds_error=False, fill_value=(0.0, 1.0)
        )(threshold)
    )
    return 1.0 - cdf_at_thr


def prophet_prob_above(prophet_row: pd.Series, threshold: float) -> float:
    """Estimate P(price > threshold) from Prophet's Gaussian-like interval."""
    import scipy.stats
    # Prophet CI is ~95% → σ ≈ (upper - lower) / (2 * 1.96)
    sigma = (prophet_row["yhat_upper"] - prophet_row["yhat_lower"]) / (2 * 1.96)
    if sigma <= 0:
        return 0.0 if threshold > prophet_row["yhat"] else 1.0
    return float(1.0 - scipy.stats.norm.cdf(threshold, loc=prophet_row["yhat"], scale=sigma))


binary_rows = []
for origin in ORIGINS:
    key   = origin.strftime("%Y-%m-%d")
    ctx   = ORIGIN_CONTEXTS[key]
    thr   = ctx["threshold_usd"]
    _, actual_price = resolution_price(price_df, origin)
    actual_event = int(actual_price > thr)

    p_row = prophet_row_at_origin(prophet_df, origin)
    p_prob = prophet_prob_above(p_row, thr)

    for tag in ["bare", "context"]:
        sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag)]
        l_prob = prob_above_threshold(sub, thr) if not sub.empty else float("nan")
        binary_rows.append({
            "origin": key,
            "label": ctx["label"],
            "threshold": thr,
            "actual_price": actual_price,
            "actual_event": actual_event,
            "method": f"LLMP — {tag}",
            "prob": l_prob,
            "brier": (l_prob - actual_event) ** 2,
        })

    binary_rows.append({
        "origin": key,
        "label": ctx["label"],
        "threshold": thr,
        "actual_price": actual_price,
        "actual_event": actual_event,
        "method": "Prophet",
        "prob": p_prob,
        "brier": (p_prob - actual_event) ** 2,
    })

binary_df = pd.DataFrame(binary_rows)
print(binary_df[["label", "threshold", "actual_price", "method", "prob", "brier"]].to_string(index=False))

In [ ]:
METHOD_COLORS = {
    "Prophet":         CLR_PROPHET,
    "LLMP — bare":     CLR_LLMP_BARE,
    "LLMP — context":  CLR_LLMP_CTX,
}
LABELS = [ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in ORIGINS]

fig = psp.make_subplots(
    rows=1, cols=3,
    subplot_titles=LABELS,
    shared_xaxes=True,
)

for col, origin in enumerate(ORIGINS, start=1):
    key = origin.strftime("%Y-%m-%d")
    ctx = ORIGIN_CONTEXTS[key]
    sub = binary_df[binary_df["origin"] == key].sort_values("method")
    actual_event = int(sub.iloc[0]["actual_event"])

    for _, row in sub.iterrows():
        method = row["method"]
        prob   = row["prob"]
        clr    = METHOD_COLORS.get(method, "#888")
        fig.add_trace(
            go.Bar(
                x=[prob],
                y=[method],
                orientation="h",
                marker_color=clr,
                marker_opacity=0.85,
                text=[f"{prob:.0%}"],
                textposition="outside",
                name=method if col == 1 else None,
                showlegend=(col == 1),
                legendgroup=method,
            ),
            row=1, col=col,
        )

    # Outcome annotation
    outcome_label = f"Outcome: {'YES' if actual_event else 'NO'}  (actual=${sub.iloc[0]['actual_price']:.0f})"
    outcome_clr   = CLR_ACTUAL if actual_event else CLR_CONFLICT
    fig.add_vline(
        x=float(actual_event),
        line=dict(color=outcome_clr, dash="dash", width=1.5),
        annotation_text=outcome_label,
        annotation_position="top left" if actual_event else "top right",
        annotation_font_size=10,
        row=1, col=col,
    )

    fig.update_xaxes(
        title_text=f"P(WTI > ${ctx['threshold_usd']:.0f})",
        range=[0, 1.1],
        tickformat=".0%",
        row=1, col=col,
    )

fig.update_layout(
    title=dict(
        text="P(WTI > Threshold in 30 Days) — by Method and Origin",
        font=dict(size=15),
    ),
    height=320,
    width=1100,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.20),
    barmode="group",
    margin=dict(t=60, b=80),
)
fig.show()

---

## Act 7 — Causal: What Forces Is the Model Anchoring On?

The trajectory and binary questions evaluate the *numbers*. This section looks at
*why* the LLMP-with-context forecasts differ from the bare LLMP.

Because the LLMP is a "direct prompt" method — it does not emit chain-of-thought —
we can't inspect its reasoning directly. But we can audit the *context we gave it*
and ask what a well-calibrated model *should* weight, then compare to what the
forecast actually did.

**This is also the setup for the Track 2 agent argument.** The LLMP here is already
doing something useful: it takes context as input and shifts its distribution accordingly.
A full Forecasting Analyst Agent would go further — it would *retrieve* that context
automatically, run code to validate signals, build scenarios, and explain its reasoning.

In [ ]:
# For each origin: show the median-forecast shift (context vs. bare) alongside
# a structured breakdown of the context signals that plausibly drove the shift.

DRIVER_TABLE: dict[str, list[dict]] = {
    "2026-01-05": [
        {"signal": "OPEC+ supply cuts maintained",      "direction": "↑ price",  "strength": "Moderate"},
        {"signal": "US-Iran proxy tensions elevated",   "direction": "↑ price",  "strength": "Moderate"},
        {"signal": "Insurance premiums rising +15%",   "direction": "↑ price",  "strength": "Weak"},
        {"signal": "US inventories 8% below avg",       "direction": "↑ price",  "strength": "Moderate"},
        {"signal": "WTI NYMEX mild backwardation",      "direction": "→ neutral", "strength": "Weak"},
        {"signal": "Soft global demand signals",        "direction": "↓ price",  "strength": "Moderate"},
    ],
    "2026-02-02": [
        {"signal": "Gulf of Oman naval incident",        "direction": "↑ price",  "strength": "Strong"},
        {"signal": "Tanker traffic -15% WoW",           "direction": "↑ price",  "strength": "Strong"},
        {"signal": "Goldman target revised $70–85",     "direction": "↑ price",  "strength": "Moderate"},
        {"signal": "OPEC+ emergency consultation",      "direction": "↑ price",  "strength": "Moderate"},
        {"signal": "Brent/WTI spread widening",         "direction": "↑ price",  "strength": "Moderate"},
        {"signal": "Iranian assets near Hormuz",        "direction": "↑ price",  "strength": "Strong"},
    ],
    "2026-03-02": [
        {"signal": "US strikes on Iranian infrastructure", "direction": "↑ price", "strength": "Very strong"},
        {"signal": "Strait of Hormuz partial blockade",   "direction": "↑ price", "strength": "Very strong"},
        {"signal": "IEA emergency session called",         "direction": "↑ price", "strength": "Strong"},
        {"signal": "Saudi force majeure declarations",     "direction": "↑ price", "strength": "Strong"},
        {"signal": "Goldman target $95–115",               "direction": "↑ price", "strength": "Strong"},
        {"signal": "Backwardation — market expects resolution", "direction": "↓ long-term", "strength": "Moderate"},
    ],
}

STRENGTH_ORDER = {"Very strong": 4, "Strong": 3, "Moderate": 2, "Weak": 1}

# Compute median shift at horizon 21 (the 30-day target)
shifts = {}
for origin in ORIGINS:
    key = origin.strftime("%Y-%m-%d")
    sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["horizon"] == HORIZON_B)]
    bare_med = float(sub[sub["context_tag"] == "bare"]["median"].iloc[0])    if not sub[sub["context_tag"] == "bare"].empty else float("nan")
    ctx_med  = float(sub[sub["context_tag"] == "context"]["median"].iloc[0]) if not sub[sub["context_tag"] == "context"].empty else float("nan")
    _, actual = resolution_price(price_df, origin)
    shifts[key] = {"bare": bare_med, "ctx": ctx_med, "actual": actual, "shift": ctx_med - bare_med}


fig = psp.make_subplots(
    rows=1, cols=3,
    subplot_titles=[ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in ORIGINS],
    shared_xaxes=False,
    horizontal_spacing=0.10,
)

DIRECTION_COLORS = {"↑ price": "#2ca02c", "↓ price": "#d62728", "↓ long-term": "#d62728", "→ neutral": "#636363"}

for col, origin in enumerate(ORIGINS, start=1):
    key     = origin.strftime("%Y-%m-%d")
    drivers = DRIVER_TABLE[key]
    s       = shifts[key]

    # Signal bars: strength as bar length, direction as colour
    for drv in drivers:
        bar_val = STRENGTH_ORDER[drv["strength"]] * (1 if drv["direction"].startswith("↑") else -1)
        fig.add_trace(
            go.Bar(
                x=[bar_val],
                y=[drv["signal"]],
                orientation="h",
                marker_color=DIRECTION_COLORS.get(drv["direction"], "#888"),
                marker_opacity=0.75,
                text=[f"{drv['direction']} ({drv['strength']})"],
                textposition="outside" if bar_val > 0 else "outside",
                showlegend=False,
            ),
            row=1, col=col,
        )

    # Annotation: median shift
    shift_label = (
        f"Median shift: +${s['shift']:.1f}  "
        f"(bare ${s['bare']:.0f} → ctx ${s['ctx']:.0f}, actual ${s['actual']:.0f})"
    )
    fig.add_annotation(
        x=0, y=-0.18,
        xref=f"x{col}", yref=f"y{col} domain",
        text=shift_label,
        showarrow=False,
        font=dict(size=9, color="#333"),
        xanchor="center",
    )

    fig.update_xaxes(
        title_text="Signal strength (→ = upward price pressure)",
        tickvals=[-4, -3, -2, -1, 0, 1, 2, 3, 4],
        ticktext=["Very strong ↓", "Strong ↓", "Mod ↓", "Weak ↓", "0", "Weak ↑", "Mod ↑", "Strong ↑", "Very strong ↑"],
        row=1, col=col,
    )

fig.update_layout(
    title=dict(
        text="Context Signal Audit — What Would a Well-Calibrated Model Weigh?",
        font=dict(size=15),
    ),
    height=500,
    width=1200,
    template="plotly_white",
    margin=dict(t=70, b=80),
)
fig.show()

### What this audit shows

The signal tables above represent the *kind of reasoning* a well-informed analyst
would apply. All of these signals were publicly available at each origin date.

The LLMP-with-context model receives this information as plain text and is asked to
continue the price trajectory. It has no tools, no ability to verify claims, no
access to proprietary data. It is, in a sense, the *minimum viable context-aware
forecaster*.

**The Track 2 Forecasting Analyst Agent would go further:**

| Capability | LLMP with hardcoded context (this notebook) | Forecasting Analyst Agent (Track 2) |
|---|---|---|
| Access to geopolitical context | Yes — but manually curated | Yes — retrieved automatically via news search |
| Futures curve analysis | Context snippet only | Queries DataService, plots term structure, runs code |
| Scenario branching | No | Yes — "blockade lasts 30 days" vs. "resolves in 2 weeks" |
| Explanation of forecast | Implicit in context shift | Explicit narrative: sources, assumptions, confidence |
| Interaction | None | Conversational Q&A |

> *"The LLMP with context is already seeing something Prophet cannot. 
> The Analyst Agent is the version that knows what to look for and explains why."*

---

## Evaluation Summary

Point MAE at the 30-day resolution horizon, and directional accuracy
(did the forecast median correctly call whether price was higher or lower
than the origin price?).

In [ ]:
eval_rows = []

for origin in ORIGINS:
    key = origin.strftime("%Y-%m-%d")
    _, actual = resolution_price(price_df, origin)
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    true_direction = actual > origin_price

    # Prophet
    p_row = prophet_row_at_origin(prophet_df, origin)
    p_mae = abs(p_row["yhat"] - actual)
    eval_rows.append({
        "Origin":     ORIGIN_CONTEXTS[key]["label"],
        "Method":     "Prophet",
        "Forecast":   f"${p_row['yhat']:.1f}",
        "Actual":     f"${actual:.1f}",
        "MAE ($)": f"{p_mae:.1f}",
        "Dir. correct": "✓" if (p_row["yhat"] > origin_price) == true_direction else "✗",
        "Inside CI": str(p_row["inside_ci"]),
    })

    # LLMP
    for tag in ["bare", "context"]:
        sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag) & (llmp_df["horizon"] == HORIZON_B)]
        if sub.empty:
            continue
        med = float(sub.iloc[0]["median"])
        mae = abs(med - actual)
        eval_rows.append({
            "Origin":     ORIGIN_CONTEXTS[key]["label"],
            "Method":     f"LLMP — {tag}",
            "Forecast":   f"${med:.1f}",
            "Actual":     f"${actual:.1f}",
            "MAE ($)": f"{mae:.1f}",
            "Dir. correct": "✓" if (med > origin_price) == true_direction else "✗",
            "Inside CI": "—",
        })

eval_df = pd.DataFrame(eval_rows)
eval_df

### Binary evaluation — Brier scores

In [ ]:
brier_pivot = (
    binary_df
    .assign(prob_pct=lambda d: (d["prob"] * 100).round(1))
    .assign(brier_str=lambda d: d["brier"].round(3).astype(str))
    .pivot_table(
        index="label",
        columns="method",
        values=["prob_pct", "brier_str"],
        aggfunc="first",
    )
)
brier_pivot.columns = [f"{col[1]} — {col[0]}" for col in brier_pivot.columns]
brier_pivot.index.name = "Origin"

# Add outcome column
outcomes = {v["label"]: ("YES" if resolution_price(price_df, pd.Timestamp(k))[1] > v["threshold_usd"] else "NO")
            for k, v in ORIGIN_CONTEXTS.items()}
brier_pivot.insert(0, "Threshold exceeded?", [outcomes[i] for i in brier_pivot.index])
brier_pivot